# Task inference — data bundle

Builds the instance store: gold + Group C v1 (test, **no labels**), silver pool + Rdatasets + Group C v2 (training, silver labels from `RULES.md`). Every instance is written to Drive as soon as it is built; after a disconnect, run cells 1–3 again and then the cell that was running.

**Runtime:** CPU is enough (no GPU). About 1 h in total.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_data'   # change if you uploaded elsewhere
import os
for f in ('config.json', 'data_lib.py', 'inputs/pool.csv', 'inputs/gold_key.csv', 'inputs/group_c_v1/key.csv'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
print('ok')

## 2. Install (pinned OpenML client, as used for the pool)

In [ ]:
%pip install -q openml==0.15.1

## 3. Load the library

In [ ]:
import sys, importlib
sys.path.insert(0, BASE)
import data_lib
importlib.reload(data_lib)
b = data_lib.Bundle(BASE)
print(b.pool.status.value_counts().to_dict())

## 4. Smoke test (~2 min, separate store under /content)

Two gold tables, two silver tables with their transforms, and the Rdatasets parser on the COUNT package.

In [ ]:
s = data_lib.Bundle(BASE, store='/content/smoke_store')
s.build_gold(only={'I002', 'I005'})
s.build_silver_pool(limit=2)
s.rdatasets_scan(packages=['COUNT'])
s.build_rdatasets(limit=3)
s.finalize()

## 5. Test sets: 56 gold + 20 Group C v1 (~10 min)

In [ ]:
b.build_gold()
b.build_group_c_v1()

## 6. Silver pool: 123 targets + Group C v2 transforms (~30 min)

In [ ]:
b.build_silver_pool()

## 7. Rdatasets: scan every documented example, then build the selected targets (~20 min)

In [ ]:
cand = b.rdatasets_scan()
print(len(cand), 'candidates')
b.build_rdatasets()

## 8. Manifest, checks and zip

Download `task_inference_store.zip` from the Drive folder and send it back (I unzip it into `data/store/`).

In [ ]:
man = b.finalize(zip_path=BASE + '/task_inference_store.zip')